In [1]:
# 종합 프로젝트용 새 데이터 생성 — 베이커리 체인 "빵셀러" 운영 데이터
!pip install numpy pandas matplotlib seaborn pyarrow -q

import os
import platform
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(13)
n_days = 90
n_stores = 6
n_rows = n_days * n_stores * 4   # 매장 x 일자 x 시간대(4구간)

stores = [f"S{i:02d}" for i in range(1, n_stores + 1)]
items = ["크루아상", "식빵", "케이크", "샌드위치", "쿠키"]

bakery = pd.DataFrame({
    "store_id": np.tile(stores, n_rows // n_stores)[:n_rows],
    "date_str": np.random.choice([
        "2025-04-01", "2025/04/01", "20250401",
        "2025-05-15", "2025/05/15", "20250515",
        "2025-06-20", "2025/06/20", "20250620",
    ], n_rows),
    "item": np.random.choice(items, n_rows),
    "quantity": np.random.choice([1, 1, 2, 2, 3, 5, 50], n_rows),  # 50은 의심값
    "unit_price": np.random.choice([3500, 4500, 5500, 8500, 12000], n_rows),
})
bakery["revenue"] = bakery["quantity"] * bakery["unit_price"]

# 오염 심기
bakery.loc[np.random.choice(n_rows, 60, replace=False), "revenue"] = np.nan
bakery.loc[5, "revenue"] = -45000  # 환불 또는 실수
bakery.loc[bakery.sample(10, random_state=1).index, "store_id"] = " S01 "  # 공백
bakery.loc[bakery.sample(8, random_state=2).index, "item"] = "케익"        # 표기 혼재('케이크' vs '케익')
bakery = pd.concat([bakery, bakery.iloc[[0, 1, 2, 3]]], ignore_index=True)   # 중복 4건

print("빵셀러 데이터 생성 완료:", bakery.shape)
bakery.head()

빵셀러 데이터 생성 완료: (2164, 6)


,store_id,date_str,item,quantity,unit_price,revenue
0,S01,20250401,쿠키,2,12000,24000.0
1,S02,2025-04-01,케이크,2,12000,24000.0
2,S03,2025-04-01,쿠키,1,5500,5500.0
3,S04,2025-06-20,쿠키,2,8500,17000.0
4,S05,20250401,샌드위치,5,4500,22500.0


In [ ]:
# 예제: 품질 리포트 함수 v2 — 수치형 컬럼에 IQR 이상치 비율을 추가
def quality_report_full(df: pd.DataFrame, name: str = "df") -> pd.DataFrame:
    '''v1에 수치형 이상치 비율(IQR)과 의심 타입 컬럼 표시를 추가합니다.'''
    n_rows = len(df)
    base = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True),
    })

    # IQR 이상치 비율 (수치형 컬럼만)
    outlier_pct = {}
    for col in df.select_dtypes(include="number").columns:  # ★ 개념③ select_dtypes: 수치형 컬럼만 선택
        s = df[col].dropna()
        s = df[col].dropna()
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        outlier_pct[col] = ((s < lo) | (s > hi)).mean() * 100
    base["outlier_pct_iqr"] = pd.Series(outlier_pct).round(2) # ★ v2에서 새로 추가된 열 ①: 이상치 비율

    # object 컬럼이 실제로는 날짜로 파싱되는지 의심 표시
    suspicious_datetime = []
    for col in df.select_dtypes(include="object").columns:  # ★ 개념③ select_dtypes: 문자열(object) 컬럼만 선택
        try:
            parsed = pd.to_datetime(df[col], errors="coerce") # ★ 개념② to_datetime coerce 트릭: 실패하면 NaT로 처리
            if parsed.notna().mean() > 0.8:
                suspicious_datetime.append(col)
        except Exception:
            pass
    base["maybe_datetime"] = base.index.isin(suspicious_datetime) # ★ v2에서 새로 추가된 열 ②: 날짜 의심 표시

    print(f"[품질 리포트(완전판)] {name}")
    print(f"  행 수: {n_rows:,}  /  열 수: {len(df.columns)}")
    print(f"  완전 중복 행: {df.duplicated().sum()}건")
    if suspicious_datetime:
        print(f"  📌 날짜로 보이는 object 컬럼: {suspicious_datetime}")
    return base


qr_orders_full = quality_report_full(orders_csv, "orders_csv")
qr_orders_full

시나리오 1 — 품질 진단 (Part 2 적용)
quality_report_full() 함수를 bakery에 그대로 적용해 다음을 답하세요.

In [ ]:
# 시나리오 1 — 모범 답안
qr_bakery = quality_report_full(bakery, "bakery")
qr_bakery

[품질 리포트(완전판)] bakery
  행 수: 2,164  /  열 수: 6
  완전 중복 행: 321건
              dtype  missing  ...  outlier_pct_iqr  maybe_datetime
store_id     object        0  ...              NaN           False
date_str     object        0  ...              NaN           False
item         object        0  ...              NaN           False
quantity      int64        0  ...            14.56           False
unit_price    int64        0  ...             0.00           False
revenue     float64       60  ...            17.30           False

[6 rows x 6 columns]


▸
결측률이 가장 높은 컬럼은? = revenue
▸
maybe_datetime이 True로 잡힌 컬럼이 있나요? = 없음
▸
수치형 중 IQR 이상치 비율이 가장 높은 컬럼은? = revenue

읽는 법: date_str이 maybe_datetime=True로 잡혔는지 확인하세요. quantity에는 50이라는 큰 값이 있어 IQR 이상치 비율이 높게 나옵니다. revenue에 결측이 있어 결측률이 가장 높습니다.

시나리오 2 — 정제 파이프라인 (Part 3·6 적용)
다음 단계의 함수를 작성하고 .pipe()로 묶어 bakery_clean을 만드세요.

In [2]:
# 시나리오 2 — 모범 답안
def b_dedup(df): return df.drop_duplicates().reset_index(drop=True) # 완전 중복 제거
def b_strip_store(df): return df.assign(store_id=df["store_id"].str.strip()) # store_id의 앞뒤 공백 제거
def b_unify_item(df): return df.assign(item=df["item"].replace({"케익": "케이크"})) # item의 '케익' → '케이크'로 통일
def b_parse_date(df): return df.assign(
    date=pd.to_datetime(df["date_str"], format="mixed", errors="coerce")
).drop(columns=["date_str"]) # date_str을 date 컬럼(datetime)으로 파싱

refunds_bakery = bakery[bakery["revenue"] < 0].copy() # revenue < 0인 행은 refunds_bakery로 분리

bakery_clean = (
    bakery
    .pipe(b_dedup)
    .pipe(b_strip_store)
    .pipe(b_unify_item)
    .pipe(b_parse_date)
    .pipe(lambda d: d[d["revenue"] >= 0])    # 환불 분리 후 본 분석에서 제외
    .pipe(lambda d: d.dropna(subset=["revenue"])) # revenue 결측 행 제거
    .reset_index(drop=True)
)

print(f"정제 전: {bakery.shape}  →  정제 후: {bakery_clean.shape}")
print(f"환불 보관: {len(refunds_bakery)}건")
print(f"item 종류: {bakery_clean['item'].unique()}")

정제 전: (2164, 6)  →  정제 후: (1784, 6)
환불 보관: 1건
item 종류: <ArrowStringArray>
['쿠키', '케이크', '샌드위치', '크루아상', '식빵']
Length: 5, dtype: str


시나리오 3 — 변환 + Parquet 저장 (Part 4·5 적용)
bakery_clean을 다음 두 KPI 표로 변환한 뒤, 각각 Parquet으로 저장하세요.

▸
월별·매장별 매출 합계 (행=매장, 열=월)
▸
상품별 평균 단가·총매출

In [5]:
# 시나리오 3 — 모범 답안

from pathlib import Path

OUT_DIR = Path("ai-data-bootcamp/D009/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# KPI 1: 월별·매장별 매출 합계 (wide)
bakery_clean["month"] = bakery_clean["date"].dt.to_period("M").astype(str)
store_month = (
    bakery_clean.groupby(["store_id", "month"])["revenue"].sum().unstack(fill_value=0).round(0)
)
print("월별·매장별 매출:")
display(store_month)

# KPI 2: 상품별 평균 단가·총매출
item_kpi = (
    bakery_clean.groupby("item")
    .agg(avg_price=("unit_price", "mean"), total_revenue=("revenue", "sum"), n_orders=("revenue", "count"))
    .round(0)
    .sort_values("total_revenue", ascending=False)
)
print("\n상품별 KPI:")
display(item_kpi)

# Parquet 저장
store_month.to_parquet(OUT_DIR / "bakery_store_month.parquet")
item_kpi.to_parquet(OUT_DIR / "bakery_item_kpi.parquet")
print(f"\nParquet 저장 완료: {OUT_DIR.resolve()}")



월별·매장별 매출:


month,2025-04,2025-05,2025-06
store_id,,,
S01,6834000.0,7243500.0,5884000.0
S02,7531500.0,6768500.0,7996500.0
S03,7424500.0,4548000.0,7297500.0
S04,6471500.0,7105000.0,6744500.0
S05,8681500.0,4102000.0,4960000.0
S06,9156000.0,5095500.0,6796500.0



상품별 KPI:


,avg_price,total_revenue,n_orders
item,,,
쿠키,6878.0,26690000.0,365
샌드위치,6555.0,26297000.0,370
케이크,6859.0,23788500.0,359
식빵,6555.0,22997500.0,355
크루아상,6946.0,20867500.0,335



Parquet 저장 완료: C:\Users\황세원\Desktop\아이펠 작업파일\ai-data-bootcamp\D009\ai-data-bootcamp\D009\outputs


In [ ]:
# 결측/중복/이상치 한 번에
qr_bakery = quality_report_full(bakery, "bakery")
display(qr_bakery)

# 표기 혼재는 문자열 컬럼마다 개별 확인
for col in bakery.select_dtypes(include="object").columns:
    print(f"--- {col} ---")
    print(bakery[col].value_counts())
    print()

              dtype  missing  missing_pct  n_unique  outlier_pct_iqr  maybe_datetime
store_id     object        0         0.00         7              NaN           False
date_str     object        0         0.00         9              NaN           False
item         object        0         0.00         6              NaN           False
quantity      int64        0         0.00         5            14.56           False
unit_price    int64        0         0.00         5             0.00           False
revenue     float64       60         2.77        26            17.30           False
--- store_id ---
store_id
S02      361
S04      361
S01      360
S03      358
S05      357
S06      357
 S01      10
Name: count, dtype: int64

--- date_str ---
date_str
20250401      261
2025/05/15    256
20250620      255
2025-04-01    242
2025-06-20    238
2025-05-15    233
20250515      231
2025/06/20    228
2025/04/01    220
Name: count, dtype: int64

--- item ---
item
쿠키      446
샌드위치    443
식빵      435
케이크     417
크루아상    415
케익        8
Name: count, dtype: int64

# 코드 퀴즈
bakery_clean에서 item이 '케이크'이고 revenue >= 10000인 행만 골라서 store_id별 평균 revenue를 구한 뒤, Parquet으로 저장하세요.

In [6]:
# 코드 퀴즈 — 모범 답안
cake_kpi = (
    bakery_clean
    .query("item == '케이크' and revenue >= 10000")
    .groupby("store_id")["revenue"]
    .mean()
    .round(0)
    .reset_index(name="avg_cake_revenue")
)
print(cake_kpi)

cake_kpi.to_parquet(OUT_DIR / "cake_kpi.parquet", index=False)
print(f"\n저장 완료: {OUT_DIR / 'cake_kpi.parquet'}")

  store_id  avg_cake_revenue
0      S01           81722.0
1      S02          134550.0
2      S03           99488.0
3      S04           85946.0
4      S05          104395.0
5      S06           71805.0

저장 완료: ai-data-bootcamp\D009\outputs\cake_kpi.parquet


In [8]:
%pip install tabulate -q

Note: you may need to restart the kernel to use updated packages.


In [9]:
# refunds 변수 먼저 준비 (음수 revenue 분리)
refunds = bakery[bakery["revenue"] < 0].copy()

# 결정 로그 기록
decisions = []

def log_decision(step: str, action: str, reason: str, result: str):
    decisions.append({"step": step, "action": action, "reason": reason, "result": result})

log_decision(
    "1. 중복 제거",
    "완전 중복 행 drop",
    "동일 키 동일 값은 시스템 입력 오류로 간주",
    f"{bakery.duplicated().sum()}건 제거",
)
log_decision(
    "2. store_id 정제",
    "strip()으로 앞뒤 공백 제거",
    "같은 매장이 표기 차이로 다른 값으로 집계되는 것 방지",
    "표기 통일 완료",
)
log_decision(
    "3. item 표기 통일",
    "'케익' → '케이크'로 replace",
    "같은 상품이 다른 항목으로 집계되는 것 방지",
    "표기 통일 완료",
)
log_decision(
    "4. revenue 음수 처리",
    "음수 revenue를 별도 변수(refunds)로 분리 보관",
    "환불 등 정상 이벤트일 수 있어 삭제 대신 분리 보관, 매출 분석에서는 제외",
    f"{len(refunds)}건 분리",
)
log_decision(
    "5. revenue 결측 처리",
    "결측 행 제거",
    "매출 합산이 분석 목적이라 결측을 채우면 실제 없었던 매출을 지어내는 왜곡 발생",
    f"{bakery['revenue'].isna().sum()}건 제거",
)

decision_log = pd.DataFrame(decisions)
display(decision_log)

# 마크다운 표로 출력 (README나 보고서 셀에 복사해서 붙여넣기용)
print("\n--- 마크다운 표 ---")
print(decision_log.to_markdown(index=False))

,step,action,reason,result
0,1. 중복 제거,완전 중복 행 drop,동일 키 동일 값은 시스템 입력 오류로 간주,321건 제거
1,2. store_id 정제,strip()으로 앞뒤 공백 제거,같은 매장이 표기 차이로 다른 값으로 집계되는 것 방지,표기 통일 완료
2,3. item 표기 통일,'케익' → '케이크'로 replace,같은 상품이 다른 항목으로 집계되는 것 방지,표기 통일 완료
3,4. revenue 음수 처리,음수 revenue를 별도 변수(refunds)로 분리 보관,"환불 등 정상 이벤트일 수 있어 삭제 대신 분리 보관, 매출 분석에서는 제외",1건 분리
4,5. revenue 결측 처리,결측 행 제거,매출 합산이 분석 목적이라 결측을 채우면 실제 없었던 매출을 지어내는 왜곡 발생,60건 제거



--- 마크다운 표 ---
| step                 | action                                        | reason                                                                        | result         |
|:---------------------|:----------------------------------------------|:------------------------------------------------------------------------------|:---------------|
| 1. 중복 제거         | 완전 중복 행 drop                             | 동일 키 동일 값은 시스템 입력 오류로 간주                                     | 321건 제거     |
| 2. store_id 정제     | strip()으로 앞뒤 공백 제거                    | 같은 매장이 표기 차이로 다른 값으로 집계되는 것 방지                          | 표기 통일 완료 |
| 3. item 표기 통일    | '케익' → '케이크'로 replace                   | 같은 상품이 다른 항목으로 집계되는 것 방지                                    | 표기 통일 완료 |
| 4. revenue 음수 처리 | 음수 revenue를 별도 변수(refunds)로 분리 보관 | 환불 등 정상 이벤트일 수 있어 삭제 대신 분리 보관, 매출 분석에서는 제외       | 1건 분리       |
| 5. revenue 결측 처리 | 결측 행 제거                                  | 매출 합산이 분석 목적이라 결측을 채우면 실제 없었던 매출을 지어내는 왜곡 발생 | 60


bakery_clean을 Parquet으로 저장하고, 같은 데이터를 CSV로도 저장해 용량 차이를 한 줄로 적었다.

In [12]:
import os

bakery_clean.to_csv(OUT_DIR / "bakery_clean.csv", index=False)
bakery_clean.to_parquet(OUT_DIR / "bakery_clean.parquet", index=False)

csv_size = os.path.getsize(OUT_DIR / "bakery_clean.csv")
parquet_size = os.path.getsize(OUT_DIR / "bakery_clean.parquet")

print(f"CSV: {csv_size:,} bytes / Parquet: {parquet_size:,} bytes")

CSV: 87,735 bytes / Parquet: 9,350 bytes


In [11]:
from pathlib import Path
OUT_DIR = Path("outputs")   # 상대경로 짧게
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUT_DIR:", OUT_DIR.resolve())

OUT_DIR: C:\Users\황세원\Desktop\아이펠 작업파일\ai-data-bootcamp\D009\outputs
